# Baseline Evaluation — Restaurant Cold-Start Recommendations

This notebook evaluates and visualizes baseline model performance across all three evaluation splits:
- **Warm**: Both user and restaurant seen in training
- **Cold Restaurant**: Restaurant held out entirely from training
- **Cold User**: User held out entirely from training

**Baselines:**
- `Random`: Uniform random scoring (absolute floor — Hit@5 ≈ 5%)
- `Popularity`: Score by review count (strong non-personalized baseline)

**Metrics:**
- `Hit@5`: Did the ground-truth restaurant appear in the top-5 ranked candidates?
- `NDCG@10`: Normalized Discounted Cumulative Gain at 10 (rewards higher placement)

In [ ]:
import json
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
})

## 1. Run Evaluation (Optional)

Uncomment and run the cell below to execute `evaluate.py` and generate fresh results.  
If you already have `results/eval_results.json`, skip to Section 2.

In [ ]:
# ── Adjust these paths as needed ─────────────────────────────────────
CONFIG_PATH = "configs/default.yaml"
CHECKPOINT_PATH = "../results/ablation/full_model/checkpoints/best_model.pt"
OUTPUT_PATH = "../results/ablation/full_model/eval_results.json"
N_NEGATIVES = 99
RATING_THRESHOLD = 3.0
HIT_K = 5
NDCG_K = 10

# Uncomment to run evaluation:
# result = subprocess.run(
#     [
#         "python", "scripts/evaluate.py",
#         "--config", CONFIG_PATH,
#         "--checkpoint", CHECKPOINT_PATH,
#         "--output", OUTPUT_PATH,
#         "--n-negatives", str(N_NEGATIVES),
#         "--rating-threshold", str(RATING_THRESHOLD),
#         "--hit-k", str(HIT_K),
#         "--ndcg-k", str(NDCG_K),
#     ],
#     capture_output=True, text=True,
# )
# print(result.stdout)
# if result.returncode != 0:
#     print("STDERR:", result.stderr)


## 2. Load Results

In [ ]:
RESULTS_PATH = Path(OUTPUT_PATH)

with open(RESULTS_PATH) as f:
    raw_results = json.load(f)

raw_results

In [ ]:
# Reshape into a tidy DataFrame for easier plotting
rows = []
for split, models in raw_results.items():
    for model, metrics in models.items():
        for metric_name, value in metrics.items():
            if metric_name == "n_cases":
                continue
            rows.append({
                "split": split,
                "model": model,
                "metric": metric_name,
                "value": value,
            })
        rows.append({
            "split": split,
            "model": model,
            "metric": "n_cases",
            "value": metrics.get("n_cases", 0),
        })

df = pd.DataFrame(rows)

# Pivot for a clean summary table
summary = df[df["metric"] != "n_cases"].pivot_table(
    index=["split", "model"], columns="metric", values="value"
).reset_index()

# Also grab test case counts per split
case_counts = (
    df[df["metric"] == "n_cases"]
    .drop_duplicates(subset=["split", "model"])
    .groupby("split")["value"]
    .first()
    .astype(int)
    .to_dict()
)

print("Test cases per split:")
for s, n in case_counts.items():
    print(f"  {s}: {n:,}")

summary

## 3. Grouped Bar Chart — Hit@K and NDCG@K by Split

The key visualization: how does each baseline perform across the three evaluation regimes?

In [ ]:
# Detect metric names dynamically from the results
metric_names = [m for m in df["metric"].unique() if m != "n_cases"]
split_order = [s for s in ["warm", "cold_restaurant", "cold_user"] if s in df["split"].unique()]
model_names = sorted(df["model"].unique())

COLORS = {
    "Random": "#9e9e9e",
    "Popularity": "#42a5f5",
    "TwoTower": "#66bb6a",      # ready for when you add the model
}

fig, axes = plt.subplots(1, len(metric_names), figsize=(6 * len(metric_names), 5), sharey=False)
if len(metric_names) == 1:
    axes = [axes]

bar_width = 0.25

for ax, metric in zip(axes, metric_names):
    sub = df[(df["metric"] == metric)].copy()
    x = np.arange(len(split_order))

    for i, model in enumerate(model_names):
        vals = []
        for split in split_order:
            match = sub[(sub["split"] == split) & (sub["model"] == model)]
            vals.append(match["value"].values[0] if len(match) > 0 else 0.0)

        offset = (i - (len(model_names) - 1) / 2) * bar_width
        bars = ax.bar(
            x + offset, vals, bar_width,
            label=model, color=COLORS.get(model, f"C{i}"),
            edgecolor="white", linewidth=0.5,
        )
        # Value labels on bars
        for bar, v in zip(bars, vals):
            ax.text(
                bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f"{v:.3f}", ha="center", va="bottom", fontsize=9,
            )

    ax.set_title(metric)
    ax.set_xticks(x)
    ax.set_xticklabels([s.replace("_", " ").title() for s in split_order])
    ax.set_ylabel(metric)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0, decimals=1))
    ax.set_ylim(0, None)
    ax.legend(loc="upper right", framealpha=0.9)

fig.suptitle("Baseline Performance Across Evaluation Splits", fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

## 4. Lift Over Random

How much better (or worse) is Popularity compared to the Random baseline?  
This isolates the signal from pure popularity ranking.

In [ ]:
if "Random" in model_names:
    lift_rows = []
    for split in split_order:
        for metric in metric_names:
            rand_val = df[
                (df["split"] == split) & (df["model"] == "Random") & (df["metric"] == metric)
            ]["value"].values
            rand_val = rand_val[0] if len(rand_val) > 0 else 0.0

            for model in model_names:
                if model == "Random":
                    continue
                model_val = df[
                    (df["split"] == split) & (df["model"] == model) & (df["metric"] == metric)
                ]["value"].values
                model_val = model_val[0] if len(model_val) > 0 else 0.0

                abs_lift = model_val - rand_val
                rel_lift = (abs_lift / rand_val * 100) if rand_val > 0 else float("inf")
                lift_rows.append({
                    "split": split,
                    "model": model,
                    "metric": metric,
                    "random_val": rand_val,
                    "model_val": model_val,
                    "abs_lift": abs_lift,
                    "rel_lift_pct": rel_lift,
                })

    lift_df = pd.DataFrame(lift_rows)
    display(lift_df.style.format({
        "random_val": "{:.4f}",
        "model_val": "{:.4f}",
        "abs_lift": "{:+.4f}",
        "rel_lift_pct": "{:+.1f}%",
    }))
else:
    print("Random baseline not found in results — skipping lift analysis.")

In [ ]:
# Relative lift bar chart
if "Random" in model_names and len(lift_df) > 0:
    non_random = [m for m in model_names if m != "Random"]

    fig, axes = plt.subplots(1, len(metric_names), figsize=(6 * len(metric_names), 4.5))
    if len(metric_names) == 1:
        axes = [axes]

    for ax, metric in zip(axes, metric_names):
        sub = lift_df[lift_df["metric"] == metric]
        x = np.arange(len(split_order))

        for i, model in enumerate(non_random):
            vals = []
            for split in split_order:
                match = sub[(sub["split"] == split) & (sub["model"] == model)]
                vals.append(match["rel_lift_pct"].values[0] if len(match) > 0 else 0.0)

            offset = (i - (len(non_random) - 1) / 2) * bar_width
            bars = ax.bar(
                x + offset, vals, bar_width,
                label=model, color=COLORS.get(model, f"C{i+1}"),
                edgecolor="white", linewidth=0.5,
            )
            for bar, v in zip(bars, vals):
                ax.text(
                    bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                    f"{v:+.0f}%", ha="center", va="bottom", fontsize=9,
                )

        ax.axhline(0, color="gray", linewidth=0.8, linestyle="--")
        ax.set_title(f"{metric} — Relative Lift Over Random")
        ax.set_xticks(x)
        ax.set_xticklabels([s.replace("_", " ").title() for s in split_order])
        ax.set_ylabel("Relative Lift (%)")
        ax.legend(loc="upper right", framealpha=0.9)

    fig.tight_layout()
    plt.show()

## 5. Expected vs. Observed Random Performance

Sanity check: Random should produce Hit@5 ≈ 5% and NDCG@10 ≈ ~4.3% with 100 candidates.  
Large deviations suggest a bug in the evaluation harness or negative sampling.

In [ ]:
# Theoretical expectations for 1 positive + 99 negatives
n_candidates = 100  # 1 positive + 99 negatives

# Hit@5: P(positive in top 5) = 5/100 = 0.05
expected_hit5 = 5 / n_candidates

# NDCG@10: E[NDCG@10] for single relevant item uniformly distributed
# = (1/n_candidates) * sum_{rank=1}^{10} 1/log2(rank+1)
expected_ndcg10 = (1 / n_candidates) * sum(1 / np.log2(r + 1) for r in range(1, 11))

print(f"With {n_candidates} candidates (1 positive + {n_candidates - 1} negatives):")
print(f"  Expected Hit@5:   {expected_hit5:.4f}")
print(f"  Expected NDCG@10: {expected_ndcg10:.4f}")
print()

if "Random" in model_names:
    print("Observed Random baseline results:")
    for split in split_order:
        for metric in metric_names:
            obs = df[
                (df["split"] == split) & (df["model"] == "Random") & (df["metric"] == metric)
            ]["value"].values
            if len(obs) > 0:
                expected = expected_hit5 if "Hit" in metric else expected_ndcg10
                deviation = (obs[0] - expected) / expected * 100
                status = "✓" if abs(deviation) < 20 else "⚠️"
                print(f"  {split:20s} {metric}: {obs[0]:.4f} (expected {expected:.4f}, {deviation:+.1f}%) {status}")

## 6. Cross-Split Comparison Heatmap

Compact view of all (model × split × metric) results.

In [ ]:
for metric in metric_names:
    pivot = df[df["metric"] == metric].pivot_table(
        index="model", columns="split", values="value"
    )
    # Reorder columns
    pivot = pivot[[s for s in split_order if s in pivot.columns]]

    fig, ax = plt.subplots(figsize=(6, max(2.5, 0.8 * len(pivot))))
    im = ax.imshow(pivot.values, cmap="YlGnBu", aspect="auto")

    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([s.replace("_", " ").title() for s in pivot.columns])
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)

    # Annotate cells
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            val = pivot.values[i, j]
            text_color = "white" if val > pivot.values.max() * 0.65 else "black"
            ax.text(j, i, f"{val:.4f}", ha="center", va="center",
                    fontsize=11, fontweight="bold", color=text_color)

    ax.set_title(f"{metric} — All Models × Splits")
    fig.colorbar(im, ax=ax, shrink=0.8)
    fig.tight_layout()
    plt.show()

## 7. Interpretation & Next Steps

**What to look for:**

- **Random ≈ theoretical**: Confirms the evaluation harness is working correctly (no leakage, proper shuffling).
- **Popularity > Random**: Expected — popular restaurants are popular for a reason. The size of the gap tells you how much signal is in raw popularity.
- **Warm vs. Cold Restaurant**: If Popularity degrades on cold restaurants, it's because new restaurants lack review counts — exactly the cold start problem the two-tower model should solve.
- **Warm vs. Cold User**: Popularity is non-personalized, so it shouldn't degrade much for cold users. If the two-tower model degrades here, it indicates the user tower is too dependent on collaborative signal.

**Targets for the Two-Tower model:**

| Split | Baseline to Beat | Minimum Acceptable |
|-------|-----------------|--------------------|
| Warm | Popularity | Popularity + meaningful lift |
| Cold Restaurant | Popularity | > Popularity (this is the primary goal) |
| Cold User | Popularity | ≥ Popularity |